# Temporal SRDB Regression Model (Model 3)

This notebook implements the baseline pipeline for **Module 3**: Modeling Soil Respiration (`Rs_annual`) dynamically over time using latitude, longitude, and historical climate variables.

This workflow precisely maps the baseline output built within `src/model_training/m3_srdb_regression.py`. We initialize our environment, load the processed Temporal SRDB Data, handle spatial categorical variables with One-Hot string encoding, transform the respiration target natively with logarithms, and train our XGBoost Regressor estimator to retrieve our performance baseline metrics.

## 1. Import Dependencies

To standardize module dependencies: Pandas and Numpy for matrix handling, scikit-learn for dataset splitting and metric calculation (RMSE, MAE), and the XGBoost ensemble regressor tree algorithm as our base model. Constants come from the centralized `src/config` manifest.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Scikit-Learn validation and base Model
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Give Jupyter notebook relative access back to src/
sys.path.append(str(Path.cwd().parent.parent))

# Load Data Configuration Paths
from src.config import (
    PROCESSED_FILES,
    WEIGHTS_DIR,
    PREPROCESS_DIR,
    RANDOM_SEED
)

## 2. Prepare Data Environment

From previous EDA exploration we know that `Rs_annual` must be transformed using natural logarithms for accurate regression fitting, and all seasonal variables (like `Rs_summer`, ratios, etc.) must be discarded initially to avoid 'data leakages' for pure overall annual prediction mapping.

In [2]:
data_path = PROCESSED_FILES["srdb_temporal"]
print(f"Loading features from {data_path.name}...")

try:
    df = pd.read_csv(data_path)
    
    # Prune non-existent Target outputs
    df = df.dropna(subset=['Rs_annual'])
    print(f"Loaded: Cleaned base shape {df.shape}")
    
    # A. Target variable creation + Native Log Transformation mapping
    y = np.log1p(df['Rs_annual'])
    print("✓ Applied natural log transform: log1p(Rs_annual)")

    # B. Prevent Data Leakage by eliminating components identical to target
    leakage_columns = [
        'Record_number', 'Rs_annual', 'Rs_spring', 'Rs_summer', 'Rs_autumn',
        'Rs_winter', 'Rs_growingseason', 'spring_ratio', 'summer_ratio',
        'autumn_ratio', 'winter_ratio', 'Decade'
    ]
    X_raw = df.drop(columns=[col for col in leakage_columns if col in df.columns])
    
    display(X_raw.head())

except FileNotFoundError:
    print(f"ERROR: {data_path} not found.")

Loading features from srdb_temporal_processed.csv...
Loaded: Cleaned base shape (5551, 19)
✓ Applied natural log transform: log1p(Rs_annual)


,Study_midyear,Latitude,Longitude,Biome,Ecosystem_type,MAT,MAP
0,2001.5,56.63,-99.94,Boreal,Forest,0.8,450.0
1,2001.5,56.46,-99.97,Boreal,Forest,0.8,450.0
2,2001.5,55.91,-98.98,Boreal,Forest,0.8,450.0
3,2001.5,55.86,-98.48,Boreal,Forest,0.8,450.0
4,2001.5,55.92,-98.39,Boreal,Forest,0.8,450.0


## 3. Dummy Encode and Train-test Split

Categorical texts like `Biome` & `Ecosystem_type` need integer mappings, we choose `pd.get_dummies()` for wide arrays (avoiding ordinal regression mistakes). We allocate our 80/20 dataset splits afterward.

In [3]:
# One-Hot Encoding for categorical text
try:
    X = pd.get_dummies(X_raw, columns=['Biome', 'Ecosystem_type'], drop_first=True)
    X = X.fillna(X.median())  # Impute NaNs just in case to provide dense matrix
    
    print(f"Feature Matrix Ready: {X.shape[0]} rows, {X.shape[1]} dimensional columns")
    
    # Validation Splitting
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )
    
except Exception as e:
    print(f"Error Encoding Data: {e}")

Feature Matrix Ready: 5551 rows, 27 dimensional columns


## 4. Train Model 3 Base

We configure the fundamental tree architecture with fixed state weights ensuring parallel reproducible execution.

In [4]:
# Generate Core Pipeline
model = XGBRegressor(
    n_estimators=300, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=RANDOM_SEED, 
    n_jobs=-1
)

print(f"Training Temporal Base Parameters : \n{model.get_params()}\n")
print("Starting base fit... ⏳")

# Validate against standard parameters to avoid deprecation overlaps 
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
print("Finished ✅")

Training Temporal Base Parameters : 
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 300, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}

Starting base fit... ⏳
Finished ✅


## 5. Evaluate and Check Scale Transformations

Evaluation yields logarithmic outputs. To determine the practical Carbon Flux ($gC/m2/yr$) margins we need to revert back exponentially using `.expm1()`. Both forms are checked below.

In [5]:
y_pred = model.predict(X_val)

print("="*40)
print("1. Log Scale Verification Base:")
print("="*40)

rmse_log = np.sqrt(mean_squared_error(y_val, y_pred))
mae_log  = mean_absolute_error(y_val, y_pred)
r2_log   = r2_score(y_val, y_pred)

print(f"R²   (Explained Var)  : {r2_log:.4f}")
print(f"RMSE (Log Residual)   : {rmse_log:.4f}")
print(f"MAE  (Log Absolute)   : {mae_log:.4f}")

# Exponentiate outputs back to native Carbon Flux Scale -> gC/m2/yr
y_val_exp  = np.expm1(y_val)
y_pred_exp = np.expm1(y_pred)

print("\n" + "="*40)
print("2. Raw Units Representation Context:")
print("="*40)

rmse_real = np.sqrt(mean_squared_error(y_val_exp, y_pred_exp))
mae_real  = mean_absolute_error(y_val_exp, y_pred_exp)

print(f"Original Unit RMSE : {rmse_real:.2f} gC/m2/yr")
print(f"Original Unit MAE  : {mae_real:.2f} gC/m2/yr")

1. Log Scale Verification Base:
R²   (Explained Var)  : 0.7297
RMSE (Log Residual)   : 0.3792
MAE  (Log Absolute)   : 0.2564

2. Raw Units Representation Context:
Original Unit RMSE : 442.76 gC/m2/yr
Original Unit MAE  : 210.32 gC/m2/yr


## 6. Export Baseline Config

With a surprisingly clean base $R^2$ of ~`0.72`, we dump the array objects out into `models/`. For Model 3, because we transformed string categories into arbitrary integer array column matrices (Dummies Setup), we **must** also save out the generated Column Features Header so our production endpoints map everything back evenly!

In [6]:
# Output Definitions
model_out_path    = WEIGHTS_DIR / "m3_srdb_regression_model.pkl"
features_out_path = WEIGHTS_DIR / "m3_srdb_features.pkl"

# Store binary instances using Joblib
joblib.dump(model, model_out_path)
print(f"📦 Model base saved to       : {model_out_path.name}")

# Without this array mapping, Production inference won't know which Biome 0s or 1s belong where!
joblib.dump(X.columns.tolist(), features_out_path)
print(f"📦 Feature headers saved to  : {features_out_path.name}")

📦 Model base saved to       : m3_srdb_regression_model.pkl
📦 Feature headers saved to  : m3_srdb_features.pkl
